# Data Preparation

Note: The electricity demand and weather datasets were provided separately for each Australian state.

Due to differences in data structure and quality between states, each state's dataset was independently cleaned and standardised before being combined into a single master dataset.

The cleaning process included:
- Converting datetime formats
- Handling missing values
- Standardising column names
- Removing invalid observations
- Aligning demand and weather records


## Example: NSW Data Cleaning
The following section demonstrates the cleaning workflow applied to the NSW dataset.

The same process was repeated for:
- New South Wales
- Queensland
- South Australia
- Tasmania

In [10]:
# --- 1.Load and clean NSW electricity demand data ---
import pandas as pd
import glob

# Locate and combine all NSW demand CSV files
demand_files = glob.glob(
    r"C:\Users\rohan\OneDrive\Documents\work\UNI WORK yr 1 sem 2\ads1002\Demand Data (1)\ALL_DEMAND_DATA\*_NSW1*.csv"
)
combined_df = pd.concat([pd.read_csv(f) for f in demand_files], ignore_index=True)

# Parse datetime and clean
combined_df['SETTLEMENTDATE'] = pd.to_datetime(combined_df['SETTLEMENTDATE'], errors='coerce')
combined_df.dropna(subset=['SETTLEMENTDATE'], inplace=True)
combined_df.sort_values('SETTLEMENTDATE', inplace=True)
combined_df.set_index('SETTLEMENTDATE', inplace=True)

# Keep only relevant demand columns and label the state
demand_tidy = combined_df[['TOTALDEMAND', 'RRP']].copy()
demand_tidy.rename(columns={'TOTALDEMAND': 'Total Demand'}, inplace=True)
demand_tidy['State'] = 'NSW1'

print(demand_tidy.shape)


(67192, 3)


In [12]:
# --- 2. Load and clean NSW weather data ---
weather_df = pd.read_csv(
    r"C:\Users\rohan\OneDrive\Documents\work\UNI WORK yr 1 sem 2\ads1002\Temperature Data\HM01X_Data_066062_999999999743964.txt",
    skipinitialspace=True,
    low_memory=False
)

# Combine multiple date/time columns into one proper datetime
weather_df['Datetime'] = pd.to_datetime(
    weather_df['Year Month Day Hour Minutes in YYYY'].astype(str).str.zfill(4) + '-' +
    weather_df['MM'].astype(str).str.zfill(2) + '-' +
    weather_df['DD'].astype(str).str.zfill(2) + ' ' +
    weather_df['HH24'].astype(str).str.zfill(2) + ':' +
    weather_df['MI format in Local time'].astype(str).str.zfill(2),
    errors='coerce'
)

# Drop invalid datetimes and sort
weather_df.dropna(subset=['Datetime'], inplace=True)
weather_df.set_index('Datetime', inplace=True)
weather_df.sort_index(inplace=True)

# Keep and rename relevant weather columns
weather_map = {
    'Precipitation since 9am local time in mm': 'Precipitation (mm)',
    'Air Temperature in degrees C':            'Air Temp (C)',
    'Relative humidity in percentage %':       'Humidity (%)'
}
keep_cols = [c for c in weather_map.keys() if c in weather_df.columns]
weather_tidy = weather_df[keep_cols].rename(columns=weather_map)

# Convert all to numeric
for col in weather_tidy.columns:
    weather_tidy[col] = pd.to_numeric(weather_tidy[col], errors='coerce')

print("Weather data ready:", weather_tidy.shape)


Weather data ready: (350945, 3)


In [14]:
# --- 3. Merge demand and weather data, format, and export ---
import openpyxl

# Merge on timestamp (inner join = only matching times)
merged = demand_tidy.join(weather_tidy, how='inner')

# Format datetime and reorder columns
out = merged.reset_index()
out.rename(columns={out.columns[0]: 'Datetime'}, inplace=True)
out['Datetime'] = pd.to_datetime(out['Datetime']).dt.strftime('%Y-%m-%d %H:%M')

desired_order = ['State', 'Datetime', 'Total Demand', 'RRP',
                 'Precipitation (mm)', 'Air Temp (C)', 'Humidity (%)']
out = out[desired_order]

# Export to Excel
output_path = r"C:\Users\rohan\OneDrive\Documents\work\UNI WORK yr 1 sem 2\ads1002\merged_nsw_demand_temp.xlsx"
out.to_excel(output_path, sheet_name="NSW_Demand_Temp", index=False, engine="openpyxl")

print(f"Exported clean dataset to: {output_path}")
print(out.head())


Exported clean dataset to: C:\Users\rohan\OneDrive\Documents\work\UNI WORK yr 1 sem 2\ads1002\merged_nsw_demand_temp.xlsx
  State          Datetime  Total Demand    RRP  Precipitation (mm)  \
0  NSW1  2000-01-01 01:00    6386.10167  14.06                 0.0   
1  NSW1  2000-01-01 01:30    5990.79500  14.30                 0.0   
2  NSW1  2000-01-01 02:00    5655.97667  14.28                 0.0   
3  NSW1  2000-01-01 02:30    5283.83667  14.17                 0.0   
4  NSW1  2000-01-01 03:00    5047.90167  10.87                 0.0   

   Air Temp (C)  Humidity (%)  
0          17.4          62.0  
1          16.8          73.0  
2          16.5          71.0  
3          16.4          65.0  
4          16.3          69.0  


## Final Dataset

After individual state processing, all cleaned datasets were combined into a single dataset used for exploratory analysis and machine learning modelling.